# Failure Mode Analysis

Aggregates per-image metrics from available runs and highlights hard cases.

In [1]:
import sys
from pathlib import Path

def _bootstrap_kvasir_seg_path() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / 'utils' / 'segmentation_common.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
        alt = p / 'Prototyping_reformat' / 'DatasetAnalysis' / 'Kvasir_SEG'
        if (alt / 'utils' / 'segmentation_common.py').exists():
            if str(alt) not in sys.path:
                sys.path.insert(0, str(alt))
            return alt
    raise RuntimeError('Could not locate Kvasir_SEG utils path from current working directory.')

BOOTSTRAP_ROOT = _bootstrap_kvasir_seg_path()
print('BOOTSTRAP_ROOT:', BOOTSTRAP_ROOT)

BOOTSTRAP_ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG


In [2]:
from pathlib import Path
import pandas as pd

from utils.segmentation_common import find_kvasir_seg_root

ROOT = find_kvasir_seg_root()
RUN_ROOTS = [
    ROOT / '1_classic_seg_baselines' / 'out',
    ROOT / '2_modern_segmentation' / 'out',
]
OUT_DIR = ROOT / '4_error_analysis' / 'out'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('RUN_ROOTS:', RUN_ROOTS)


ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG
RUN_ROOTS: [PosixPath('/mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/1_classic_seg_baselines/out'), PosixPath('/mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/2_modern_segmentation/out')]


In [3]:
# Load per-image test outputs if they exist
per_image_files = []
for rr in RUN_ROOTS:
    if rr.exists():
        per_image_files.extend(sorted(rr.rglob('per_image_test.csv')))

per_image_files = sorted({p.resolve() for p in per_image_files})

if not per_image_files:
    print('No per_image_test.csv files found yet. Run baseline notebooks first.')
else:
    frames = []
    for p in per_image_files:
        df = pd.read_csv(p)
        df['run_name'] = p.parent.name
        frames.append(df)

    all_df = pd.concat(frames, ignore_index=True)

    summary = (
        all_df.groupby('run_name')
        .agg(
            n=('img_id', 'count'),
            dice_mean=('dice', 'mean'),
            iou_mean=('iou', 'mean'),
            recall_mean=('recall', 'mean'),
            precision_mean=('precision', 'mean'),
        )
        .reset_index()
        .sort_values('dice_mean', ascending=False)
    )

    hard_cases = (
        all_df.sort_values(['dice', 'iou', 'recall'], ascending=True)
              .head(50)
              [['run_name', 'img_id', 'dice', 'iou', 'precision', 'recall', 'component_count', 'true_area_ratio']]
    )

    summary_path = OUT_DIR / 'model_summary.csv'
    hard_cases_path = OUT_DIR / 'hard_cases_top50.csv'
    summary.to_csv(summary_path, index=False)
    hard_cases.to_csv(hard_cases_path, index=False)

    print('Saved:', summary_path)
    print('Saved:', hard_cases_path)
    display(summary)
    display(hard_cases.head(10))


Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/4_error_analysis/out/model_summary.csv
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/4_error_analysis/out/hard_cases_top50.csv


,run_name,n,dice_mean,iou_mean,recall_mean,precision_mean
0,deeplabv3plus_resnet50_baseline,100,0.718198,0.611093,0.724707,0.800357
2,unet_resnet34_baseline,100,0.570666,0.438776,0.709143,0.574819
1,segformer_b2_finetune,100,0.474886,0.345191,0.760785,0.421988


,run_name,img_id,dice,iou,precision,recall,component_count,true_area_ratio
1,deeplabv3plus_resnet50_baseline,cju0tl3uz8blh0993wxvn7ly3,0.000000,0.000000,0.000000,0.000000,1,0.005246
79,deeplabv3plus_resnet50_baseline,cju7dqcwi2dz00850gcmr2ert,0.000000,0.000000,0.000000,0.000000,1,0.054502
90,deeplabv3plus_resnet50_baseline,cju8bj2ssrmlm0871gc2ug2rs,0.000000,0.000000,0.000000,0.000000,1,0.004778
101,unet_resnet34_baseline,cju0tl3uz8blh0993wxvn7ly3,0.000000,0.000000,0.000000,0.000000,1,0.005246
126,unet_resnet34_baseline,cju2zpw4q9vzr0801p0lysjdl,0.000000,0.000000,0.000000,0.000000,1,0.031807
133,unet_resnet34_baseline,cju34fojcctcf0799ebolbvkn,0.000000,0.000000,0.000000,0.000000,1,0.027675
253,segformer_b2_finetune,cju5k503sfa5f0871lx0rpu5y,0.000000,0.000000,0.000000,0.000000,1,0.155971
233,segformer_b2_finetune,cju34fojcctcf0799ebolbvkn,0.004504,0.002257,0.002589,0.017304,1,0.027643
251,segformer_b2_finetune,cju5hyi9yegob0755ho3do8en,0.006941,0.003483,0.005070,0.011001,2,0.088768
262,segformer_b2_finetune,cju6v6g6kvdw007552x6mb0po,0.009903,0.004976,0.007712,0.013832,2,0.145528
